# Évaluation de la Restauration Minière (Reclamation)

## Introduction
La fin de l'exploitation d'une mine impose souvent des obligations de réhabilitation (reforestations, stabilisation des sols). Ce notebook permet de vérifier scientifiquement si la végétation reprend ses droits sur les anciens sites miniers du Congo en utilisant des indices de vitalité chlorophyllienne.

## Objectifs
*   **Mesure du reboisement** : Quantifier le gain de biomasse sur les anciennes carrières.
*   **Score de réhabilitation** : Calculer un indice de réussite de la restauration par rapport à l'état initial.
*   **Suivi réglementaire** : Fournir des preuves visuelles et statistiques de la remise en état.

## Méthodologie
1.  **Setup** : Installation des bibliothèques.
2.  **Acquisition** : Comparaison (théorique ici) ou analyse d'images Sentinel-2.
3.  **Calcul NDVI** : Isolation des zones ayant une activité photosynthétique stable.
4.  **Zonage de succès** : Cartographie des secteurs restaurés vs secteurs encore stériles.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib seaborn -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Localisation des sites miniers en phase de fermeture ou déjà fermés.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition et Calcul du Succès Végétal
Nous calculons l'indice NDVI. Une valeur > 0.4 sur un ancien site minier est un indicateur fort d'une restauration végétale réussie.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données et NDVI
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B4', 'B8']), 'reclaim.tif', scale=30, region=roi)

with rasterio.open('reclaim.tif') as src: 
    data = src.read().astype(np.float32)
red, nir = data[0], data[1]
ndvi = (nir - red) / (nir + red + 1e-8)

reclamation_zone = ndvi > 0.4

plt.figure(figsize=(10, 8))
plt.imshow(reclamation_zone, cmap='RdYlGn')
plt.title('Évaluation de la Reconstruction Végétale (Vert = Succès)')
plt.axis('off')
plt.show()